# HeritageGuard AI: Policy Recommendation Agent (Colab)
Mount Carmel Church, Chimbel, Goa. Run the cells in order.

## 1. Install

In [1]:
!pip -q install -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


## 2. Upload the project zip
Upload `heritageguard-agent.zip` when asked.

In [4]:
from google.colab import files
files.upload()
!unzip -oq heritageguard-agent.zip
%cd heritageguard-agent

Saving .gitignore to .gitignore
Saving app.py to app.py
Saving HeritageGuard_Agent_Colab.ipynb to HeritageGuard_Agent_Colab.ipynb
Saving README.md to README.md
Saving requirements.txt to requirements.txt
/content/heritageguard-agent


In [5]:
!pwd

/content/heritageguard-agent


In [6]:
!ls

agent	data				 README.md
app.py	HeritageGuard_Agent_Colab.ipynb  requirements.txt


In [7]:
!ls agent data

agent:
heritageguard.py  __init__.py

data:
article_facts.json  policy_areas.json


## 3. Add your Gemini key
Get a key at aistudio.google.com. In Colab, click the key icon (Secrets) in the left bar, add a secret named `GEMINI_API_KEY`, paste the key, and switch on Notebook access. Never paste the key into a cell.

In [8]:
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
# Optional: os.environ['GEMINI_MODEL'] = 'a-model-name-from-AI-Studio'

## 4. Paste the article text
Replace the text between the quotes. The agent also uses `data/article_facts.json`.

In [9]:
ARTICLE = """
Govt boost for historic Mount Carmel site, locals for urgent conservation
Source: The Times of India (Goa)
Byline: Paul Fernandes

The article discusses the deteriorating condition of the historic Church of Our Lady of Mount Carmel in Chimbel, Goa, and the urgent need to conserve the remaining structure.

Built around 1749–1750 by the Tertiary Carmelites, the church is an example of Indo-Portuguese architecture in the Mannerist style. It once stood as an important religious and social landmark in the area. The article describes its historical significance and the threats facing the remains of the building.

The church has suffered extensive structural deterioration. Large portions of the structure and walls have collapsed, while vegetation growing around the ruins has contributed to the deterioration of the remaining facade. The site's neglected condition raises concerns about the loss of an important part of Goa's architectural and cultural heritage.

The government notified the site as a protected heritage monument in December 2020. However, protected status alone has not prevented further deterioration. Local residents and the Mount Carmel Conservation Association have called for urgent conservation measures to safeguard what remains of the church.

The article discusses proposals for structural preservation and the development of green space around the site. It also highlights the possibility of connecting Mount Carmel Church, located approximately 7 km from Old Goa, to the wider heritage tourism circuit.
"""

## 5. Run the agent

In [15]:
import os, time
from google import genai
from google.genai import types

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(timeout=30000),
)
names = [m.name.replace("models/", "") for m in client.models.list() if "flash" in m.name]
print(names)
for n in names:
    try:
        t = time.time()
        r = client.models.generate_content(model=n, contents="Say hello in five words.")
        print("WORKS:", n, f"({time.time() - t:.1f}s)")
    except Exception as e:
        print("failed:", n, str(e)[:80])

['gemini-2.5-flash', 'gemini-2.5-flash-preview-tts', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-image', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3.1-flash-image-preview', 'gemini-3.1-flash-image', 'gemini-3.1-flash-lite-image', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-3.1-flash-tts-preview', 'gemini-3.8-flash-tts', 'gemini-3.8-flash-lite-tts', 'gemini-2.5-flash-native-audio-latest', 'gemini-2.5-flash-native-audio-preview-09-2025', 'gemini-2.5-flash-native-audio-preview-12-2025', 'gemini-3.1-flash-live-preview']
failed: gemini-2.5-flash 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-
failed: gemini-2.5-flash-preview-tts 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'The requested combinat
failed: gemini-flash-latest 504

In [16]:
import os, importlib
os.environ['GEMINI_MODEL'] = 'gemini-3.5-flash'
import agent.heritageguard as h
importlib.reload(h)
result = h.run_agent(ARTICLE, progress=print)
print('Model used:', result['model'])

Step 1/3: analysing the issues
Step 2/3: generating policy options
Step 3/3: reviewing for unsupported claims
Model used: gemini-3.5-flash


## 6. View the recommendations (screenshot this for the exam)

In [17]:
from IPython.display import Markdown, display

def bl(title, items):
    return (f"**{title}**\n" + "\n".join(f"- {i}" for i in items) + "\n\n") if items else ""

a = result["analysis"]
out = "# Issue analysis\n"
out += "".join(f"- **{p['issue']}**: {p['evidence']}\n" for p in a["problems"])
out += "\n" + bl("Stakeholders", a["stakeholders"]) + bl("Priorities", a["priorities"])
out += "# Policy recommendations\n"
for p in result["policies"]:
    out += f"\n## {p['area']}: {p['recommendation']}\n{p['justification']}\n\n"
    out += bl("Evidence from the article", p["evidence_from_article"]) + bl("Assumptions", p["assumptions"])
    out += bl("Expected benefits", p["expected_benefits"]) + bl("Trade-offs", p["trade_offs"])
    out += bl("Implementation challenges", p["implementation_challenges"])
    out += bl("Success indicators", p["success_indicators"]) + bl("Evidence to verify", p["evidence_to_verify"])
out += "\n# Reviewer flags\n"
out += "".join(f"- **{f['policy']}**: {f['concern']}\n" for f in result["review_flags"]) or "No unsupported claims flagged (automated check; still review manually).\n"
display(Markdown(out))

# Issue analysis
- **Structural collapse and physical neglect of the heritage ruins**: Civil engineer status report indicates that most of the structure and adjoining walls have collapsed, debris lies around, and the property appears abandoned.
- **Instability of the facade from vegetation and natural elements**: Tree growth has destabilized the cracked facade, which continues to be endangered by neglect and weather.
- **Inability of heritage notification alone to stop decay**: The government notified the site as a protected heritage monument in December 2020, but protected status alone has not prevented ongoing deterioration.

**Stakeholders**
- Government of Goa
- Mount Carmel Conservation Association (MCCAC)
- Local residents and villagers of Chimbel
- Civil engineer Jennifer Lewis e Kamat
- Tourists and researchers

**Priorities**
- Obtaining a qualified structural assessment to inform structural preservation interventions
- Stabilizing the cracked facade and remaining boundary walls
- Developing green space around the site for rejuvenation
- Exploring potential linkages to the heritage tourism circuit 7 km from Old Goa

# Policy recommendations

## Conservation funding models: Commission a comprehensive structural assessment by qualified engineers to establish baseline costs prior to creating a dedicated funding framework for ruin stabilization and site upkeep.
In accordance with conservation guidelines, physical restoration or stabilization interventions should not proceed without a prior professional structural assessment, especially given the extensive collapse noted by civil engineer Jennifer Lewis e Kamat.

**Evidence from the article**
- A civil engineer's status report says most of the structure and adjoining walls have collapsed and debris lies around the structure.
- The civil engineer, Jennifer Lewis e Kamat, says the property seems abandoned and the church needs immediate structural intervention.
- Government first notified the extensive complex as a heritage site on 12 December 2020.

**Assumptions**
- State heritage funding mechanisms can be activated once formal technical assessments quantify structural requirements.

**Expected benefits**
- Prevents unsafe or inappropriate physical interventions on fragile ruins.
- Provides clear financial estimates for long-term stabilization and maintenance.

**Trade-offs**
- Immediate physical work is paused while technical assessments are completed.

**Implementation challenges**
- Procuring qualified heritage conservation engineers to perform detailed structural risk mapping.

**Success indicators**
- Completion of an official structural engineering status report.
- Establishment of a phased conservation budget based on assessment findings.

**Evidence to verify**
- Exact government financial allocations currently assigned to notified protected monuments in Goa.


## Community-led stewardship: Develop a formal collaborative stewardship agreement between state heritage authorities and the Mount Carmel Conservation Association for site monitoring and green space care.
Local community groups have consistently advocated for site protection and can provide continuous on-ground monitoring and maintain planned green spaces for local use.

**Evidence from the article**
- The MCCAC has pursued a long campaign for the site's heritage status and recently held a meeting.
- The MCCAC secretary says conserving the facade and boundary wall would give villagers a sit-out for prayers and Masses.
- A proposal considers structural preservation of the ruins, with some green space around for rejuvenation.

**Assumptions**
- Local community volunteers possess the capacity to conduct routine non-structural site maintenance under professional oversight.

**Expected benefits**
- Enhances local oversight to deter dumping, vandalism, or neglect.
- Fosters community ownership while providing a usable sit-out for residents.

**Trade-offs**
- Requires ongoing administrative coordination between government officials and volunteer association representatives.

**Implementation challenges**
- Establishing clear legal terms defining community maintenance boundaries without compromising statutory heritage authority.

**Success indicators**
- Formalization of a stewardship memorandum between local authorities and MCCAC.
- Sustained maintenance of designated green spaces and regular site monitoring logs.

**Evidence to verify**
- The vice president of the Mount Carmel Conservation Association (MCCAC) says a plan to conserve and beautify the site would help draw tourists. [verify]


## Heritage-tourism integration: Design a low-impact heritage route connecting Chimbel to Old Goa that highlights the unique socio-religious and architectural history of the site while controlling visitor footfall.
The site is located 7 km from Old Goa and possesses distinct historical features, making it suitable for heritage linkages that do not overwhelm fragile structural remains.

**Evidence from the article**
- The site is barely 7 km from Old Goa and could be linked to the heritage circuit.
- The place was once a base for the Tertiary Carmelites, who ran Goa's first mental health care hospital, a home for the destitute and other socio-religious activities there.
- The church ruins were built around 1749-1750 by the Tertiary Carmelites, in Indo-Portuguese Mannerist style.

**Assumptions**
- Heritage tourists visiting Old Goa would extend visits to secondary regional sites if clear interpretation and access exist.

**Expected benefits**
- Spreads heritage tourism benefits beyond Old Goa to surrounding communities.
- Increases public appreciation of Goa's early medical and social care history.

**Trade-offs**
- Increased visitor movement may disrupt the tranquil rural environment of the local neighborhood.

**Implementation challenges**
- Regulating tourist flow to ensure unstable structural ruins are not accessed or damaged.

**Success indicators**
- Integration of Chimbel into official regional heritage tourism maps.
- Installation of educational signage explaining the history of the Tertiary Carmelites.

**Evidence to verify**
- The relocation of the destitute home, because a building near the ruins was in poor condition, hastened the decay of the ruins. [verify]


## Urban-planning buffers: Establish statutory planning buffer zones around the heritage property boundaries to regulate real estate development and preserve the site's tranquil setting.
The ruins are located in a fast-urbanising landscape, requiring proactive land-use planning controls to prevent surrounding construction from encroaching on or destabilizing the monument.

**Evidence from the article**
- The ruins lie in a tranquil spot of a fast-urbanising landscape.
- Government first notified the extensive complex as a heritage site on 12 December 2020.
- Tree growth has destabilised the cracked facade, which is also endangered by neglect and natural elements.

**Assumptions**
- Local urban planning bodies hold authority to enforce special zoning restrictions adjacent to notified heritage sites.

**Expected benefits**
- Protects the visual integrity, micro-environment, and quiet setting of the historic ruins.
- Mitigates structural risks caused by heavy urban construction in close proximity.

**Trade-offs**
- Restricts private land development opportunities within the designated buffer area.

**Implementation challenges**
- Resolving potential land-use conflicts with surrounding property owners in an expanding urban area.

**Success indicators**
- Incorporation of a defined heritage buffer zone into the regional town planning master plan.
- Enforcement of height and density restrictions surrounding the site.

**Evidence to verify**
- The site gained prominence after an earlier Times of India report and a study by two researchers on its architectural and historical dimensions. [verify]


# Reviewer flags
No unsupported claims flagged (automated check; still review manually).


## 7. Save the output

In [18]:
import json
with open('output.json', 'w') as f:
    json.dump(result, f, indent=2)
files.download('output.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>